In [71]:
from vespa.package import Field, RankProfile, FirstPhaseRanking
%load_ext autoreload
%autoreload 2
import json
import mycode.vap as vap
import mycode.trace as trace

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# The goal is to have a demo application that has 10M docs
# - numeric field like timestamp in second granularity with fast-search
# - numeric field like timestamp in second granularity witout fast-search
# - numeric field with hour granularity with fast-search
# - numeric field with daily granularity with fast-search
# - single dimension embedding field for the nearestNeighbor search
# - field string type for weakAnd
# Show that a when using a field with high cardinality, the posting list preparation time is astronomical.
# Show that when using rounded timestamp the postling list preparation is tamed.
# Show that when (rounded_timestamp > X) AND (precise_timestamp > X) AND NN, then the query is super slow
# Show that (rounded_timestamp > X) AND (precise_timestamp_with_fast_search > X) just adds the posting list preparation time with no benefit for the query

In [3]:
app = vap.demo_application_package()

In [4]:
from vespa.package import Field
from vespa.package import QueryTypeField, QueryProfileType

app.get_schema("doc").add_fields(
    Field(
        name="timestamp_second_nofs",
        type="int",
        indexing="attribute",
        # no fast search
    ),
    Field(
        name="timestamp_second",
        type="int",
        indexing="attribute",
        attribute=["fast-search"],
    ),
    Field(
        name="timestamp_hour",
        type="int",
        indexing="attribute",
        attribute=["fast-search"],
    ),
    Field(
        name="timestamp_day",
        type="int",
        indexing="attribute",
        attribute=["fast-search"],
    ),
    Field(
        name="embedding",
        type="tensor<float>(x[1])",
        indexing="attribute"
    ),
    Field(
        name="lexical",
        type="string",
        indexing="index",
        index="enable-bm25"
    ),
)

app.query_profile_type = QueryProfileType(
    fields=[
        QueryTypeField(
            name="ranking.features.query(query_embedding)",
            type="tensor<float>(x[1])"
        )
    ]
)

app.get_schema("doc").rank_profiles.pop("fields")

RankProfile('fields', '0', 'unranked', None, [Function('id', 'attribute(id)', None)], ['id'], ['id'], None, None, None, None, None, None, None, None, None, None, None)

In [5]:
#| label: schema
print(app.get_schema("doc").schema_to_text)

schema doc {
    document doc {
        field id type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field timestamp_second_nofs type int {
            indexing: attribute
        }
        field timestamp_second type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field timestamp_hour type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field timestamp_day type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field embedding type tensor<float>(x[1]) {
            indexing: attribute
        }
        field lexical type string {
            indexing: index
            index: enable-bm25
        }
    }
}


In [7]:
from vespa.deployment import VespaDocker

# In case running colima on macos run the following
# !sudo ln -sf $HOME/.colima/default/docker.sock /var/run/docker.sock
vespa_docker = VespaDocker(
    container_image="vespaengine/vespa:8.672.3",
)
# Start a docker container and deploy the application package
client = vespa_docker.deploy(
    application_package=app,
)

Waiting for configuration server, 0/60 seconds...
Waiting for configuration server, 5/60 seconds...
Application is up!
Finished deployment.


In [122]:
from vespa.package import RankProfile, FirstPhaseRanking

app.get_schema("doc").add_rank_profile(RankProfile(
    name="minimal",
    first_phase=FirstPhaseRanking(expression="1", keep_rank_count=1),
))

vap.redeploy(vespa_docker, app)

Deploy status code: 200


Vespa(http://localhost, 8080)

In [63]:
from vespa.io import VespaResponse
import random
import datetime


def simulate_text():
    """
    Pics a random number of words from random numbers from 0 to 30000.
    Joins them in to a string.
    :return:
    """
    dictionary_size = 30001
    num_words = random.randint(1, 20)
    return " ".join(map(lambda n: str(n), random.sample(range(dictionary_size), num_words)))


def simulate_embedding():
    """
    Random float number between 0 and 1.
    :return:
    """
    return [random.uniform(0, 1)]


def shuf_range(start, end):
    """
    Generate a bunch of item ids, shuffle them, so that feeding is faster.
    :param start:
    :param end:
    :return:
    """
    numbers = list(range(start, end))
    random.shuffle(numbers)
    return numbers


def get_timestamp():
    """
    current timestamp in seconds as integer
    :return:
    """
    return int(datetime.datetime.now().timestamp())


def round_to_hours(timestamp):
    """
    Get first second of the hour
    :param timestamp:
    :return:
    """
    return int(timestamp / 3600) * 3600


def round_to_days(timestamp):
    """
    Get first second of the day
    :param timestamp:
    :return:
    """
    return int(timestamp / (3600 * 24)) * (3600 * 24)


def feed_call_back_with_progress(n=100000):
    cnt = 0

    def callback(response: VespaResponse, document_id: str):
        nonlocal cnt
        cnt += 1
        if (cnt % n) == 0:
            print(f"{datetime.datetime.now().isoformat()}: Already fed: {cnt} docs")
        if not response.is_successful():
            print(f"Error when feeding document {document_id}: {response.get_json()}")

    return callback

In [68]:
amount_of_docs = 10_000_000  # around four months every second
ids = shuf_range(get_timestamp(), get_timestamp() + amount_of_docs)
print("generated doc ids")
vap.feed(
    client=client,
    docs=(
        {
            "timestamp_second_nofs": i,
            "timestamp_second": i,
            "timestamp_hour": round_to_hours(i),
            "timestamp_day": round_to_days(i),
            "embedding": simulate_embedding(),
            "lexical": simulate_text()
        } for i in ids),
    feed_callback=feed_call_back_with_progress(10000)
)

generated doc ids
2026-04-16T16:29:18.856387: Already fed: 10000 docs
2026-04-16T16:29:24.515968: Already fed: 20000 docs
2026-04-16T16:29:29.542749: Already fed: 30000 docs
2026-04-16T16:29:34.480674: Already fed: 40000 docs
2026-04-16T16:29:39.326024: Already fed: 50000 docs
2026-04-16T16:29:44.223216: Already fed: 60000 docs
2026-04-16T16:29:49.170994: Already fed: 70000 docs
2026-04-16T16:29:54.139384: Already fed: 80000 docs
2026-04-16T16:29:59.224948: Already fed: 90000 docs
2026-04-16T16:30:04.186116: Already fed: 100000 docs
2026-04-16T16:30:09.135326: Already fed: 110000 docs
2026-04-16T16:30:14.014746: Already fed: 120000 docs
2026-04-16T16:30:18.954131: Already fed: 130000 docs
2026-04-16T16:30:23.880038: Already fed: 140000 docs
2026-04-16T16:30:29.000893: Already fed: 150000 docs
2026-04-16T16:30:33.901696: Already fed: 160000 docs
2026-04-16T16:30:38.805713: Already fed: 170000 docs
2026-04-16T16:30:43.667307: Already fed: 180000 docs
2026-04-16T16:30:48.605322: Already f

In [99]:
#| label: and_query

def yql_base(field):
    return f"""
                 select *
                 from sources *
                 where
                    ({field} > {get_timestamp()})
                 """


def request(field="timestamp_second"):
    return {
        "yql": yql_base(field),
        "query_str": "27110 6334 10140 22335 22040 2716",
        "input.query(query_embedding)": [0.5],
        "presentation.timing": True,
        "hits": 1,
        "timeout": "2s",
        "ranking.profile": "unranked",
    }


resp_no_filters = client.query(body=trace.add_trace(request("timestamp_second"))).json
print(trace.get_breakdown(trace.inspect_trace(resp_no_filters)))

looking into node doc[0]
┌────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp  │ event                                                       │
├────────────┼─────────────────────────────────────────────────────────────┤
│   0.099 ms │ searching for 1 hits at offset 0                            │
│   0.119 ms │ Start query setup                                           │
│   0.121 ms │ Deserialize and build query tree                            │
│   0.129 ms │ Build query execution plan                                  │
│   0.231 ms │ Optimize query execution plan                               │
│   0.238 ms │ Perform dictionary lookups and posting lists initialization │
│ 338.284 ms │ Prepare shared state for multi-threaded rank executors      │
│ 338.295 ms │ Complete query setup                                        │
│            │ (query execution happens here, analyzed below)              │
│ 980.235 ms │ returning 1 hits from total 7452374 

In [225]:
#| label: baseline_query
print(yql_base("timestamp_second"))


                 select *
                 from sources *
                 where
                    (timestamp_second > 1776433732)
                 


In [100]:
# the dictionary lookup is 338ms for the second granularity

In [273]:
resp_no_filters = client.query(body=request("timestamp_second")).json
resp_no_filters

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::4',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::4',
    'relevance': 0.0,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 10000000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 9845178},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.378, 'searchtime': 0.378, 'summaryfetchtime': 0.0}}

In [231]:
#| label: timestamp_hour_simple_filter
timestamp_hour = client.query(body=trace.add_trace(request("timestamp_hour"))).json
print(trace.get_breakdown(trace.inspect_trace(timestamp_hour)))

looking into node doc[0]
┌─────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp   │ event                                                       │
├─────────────┼─────────────────────────────────────────────────────────────┤
│    0.100 ms │ searching for 1 hits at offset 0                            │
│    0.129 ms │ Start query setup                                           │
│    0.144 ms │ Deserialize and build query tree                            │
│    0.153 ms │ Build query execution plan                                  │
│    0.261 ms │ Optimize query execution plan                               │
│    0.269 ms │ Perform dictionary lookups and posting lists initialization │
│   40.587 ms │ Prepare shared state for multi-threaded rank executors      │
│   40.596 ms │ Complete query setup                                        │
│             │ (query execution happens here, analyzed below)              │
│ 1017.026 ms │ returning 1 hits from t

In [102]:
# hourly, 34 ms. better,

In [109]:
timestamp_day = client.query(body=trace.add_trace(request("timestamp_day"))).json
print(trace.get_breakdown(trace.inspect_trace(timestamp_day)))

looking into node doc[0]
┌────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp  │ event                                                       │
├────────────┼─────────────────────────────────────────────────────────────┤
│   0.104 ms │ searching for 1 hits at offset 0                            │
│   0.129 ms │ Start query setup                                           │
│   0.130 ms │ Deserialize and build query tree                            │
│   0.151 ms │ Build query execution plan                                  │
│   0.185 ms │ Optimize query execution plan                               │
│   0.193 ms │ Perform dictionary lookups and posting lists initialization │
│  34.783 ms │ Prepare shared state for multi-threaded rank executors      │
│  34.791 ms │ Complete query setup                                        │
│            │ (query execution happens here, analyzed below)              │
│ 986.984 ms │ returning 1 hits from total 9599005 

In [ ]:
# somewhat faster but very little

In [111]:
timestamp_second_nofs = client.query(body=trace.add_trace(request("timestamp_second_nofs"))).json
print(trace.inspect_trace(timestamp_second_nofs))

┌─────────┬─────────────┐
│ total   │ 1005.000 ms │
├─────────┼─────────────┤
│ query   │ 1004.000 ms │
│ summary │    0.000 ms │
│ other   │    1.000 ms │
└─────────┴─────────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │   1002.406 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 1002.406 ms
┌───────────────┬────────────┐
│ task          │ doc[0]     │
├───────────────┼────────────┤
│ global filter │   0.000 ms │
│ ann setup     │   0.000 ms │
│ matching      │ 735.835 ms │
│ first phase   │   0.000 ms │
│ second phase  │   0.000 ms │
└───────────────┴────────────┘
looking into node doc[0]
┌─────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp   │ event                                                       │
├─────────────┼────────────

In [233]:
#| label: nofs_iteration_overhead
print(trace.get_matching_summary(trace.inspect_trace(timestamp_second_nofs)))

match profiling for thread #0 (total time was 735.835 ms)
┌─────────┬──────────┬─────────┬──────┬────────────────────────────────────────────────────────────────┐
│ seeks   │ total_ms │ self_ms │ step │ query tree                                                     │
├─────────┼──────────┼─────────┼──────┼────────────────────────────────────────────────────────────────┤
│ 4702240 │  735.835 │ 465.171 │ S    │  And[1]                                                        │
│ 4702240 │  134.666 │ 134.666 │ S    │  ├── Attribute{int32,lookup}[2] timestamp_second_nofs:<range>  │
│ 4702240 │  135.999 │ 135.999 │ N    │  └── WhiteList[3]                                              │
└─────────┴──────────┴─────────┴──────┴────────────────────────────────────────────────────────────────┘



In [145]:
# Now let's combine the hourly matching with second level matching _nofs
# AND timestamp_second_nofs > {get_timestamp()}
#  AND timestamp_second_nofs < {get_timestamp() + 5000000}
combined = client.query(body={
    **trace.add_trace(request("timestamp_second_nofs")),
    "yql": f"""
                 select *
                 from sources *
                 where
                    (timestamp_hour >= {get_timestamp()} )
                    AND (timestamp_hour <= {get_timestamp() + 1000000})
                    AND ({{targetHits: 1000, approximate: false}}
                         nearestNeighbor(embedding, query_embedding))
                 """,
    "ranking.profile": "minimal",
}).json
print(trace.inspect_trace(combined))

┌─────────┬────────────┐
│ total   │ 164.000 ms │
├─────────┼────────────┤
│ query   │ 163.000 ms │
│ summary │   0.000 ms │
│ other   │   1.000 ms │
└─────────┴────────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │    159.450 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 159.450 ms
┌───────────────┬────────────┐
│ task          │ doc[0]     │
├───────────────┼────────────┤
│ global filter │   0.000 ms │
│ ann setup     │   0.000 ms │
│ matching      │ 153.479 ms │
│ first phase   │   0.001 ms │
│ second phase  │   0.000 ms │
└───────────────┴────────────┘
looking into node doc[0]
┌────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp  │ event                                                       │
├────────────┼───────────────────────

In [143]:
#Hour
match
profiling
for thread  #0 (total time was 187.187 ms)
    ┌─────────┬──────────┬─────────┬──────┬─────────────────────────────────────────────────────┐
│ seeks   │ total_ms │ self_ms │ step │ query
tree                                          │
├─────────┼──────────┼─────────┼──────┼─────────────────────────────────────────────────────┤
│    7965 │  187.187 │  70.540 │ S    │  And[1]                                             │
│ 1000801 │   15.177 │  15.177 │ S    │  ├── Attribute
{int32, fs}[2]
timestamp_hour: < range >  │
│ 1000801 │   15.176 │  15.176 │ N    │  ├── WhiteList[3]                                   │
│ 1000800 │   86.294 │  86.294 │ N    │  └── NearestNeighbor[4]                             │
└─────────┴──────────┴─────────┴──────┴─────────────────────────────────────────────────────┘
#day
match
profiling
for thread  #0 (total time was 158.239 ms)
    ┌─────────┬──────────┬─────────┬──────┬────────────────────────────────────────────────────┐
│ seeks   │ total_ms │ self_ms │ step │ query
tree                                         │
├─────────┼──────────┼─────────┼──────┼────────────────────────────────────────────────────┤
│    7967 │  158.239 │  75.667 │ S    │  And[1]                                            │
│ 1036801 │   16.031 │  16.030 │ S    │  ├── Attribute
{int32, fs}[2]
timestamp_day: < range >  │
│ 1036801 │   16.030 │  16.030 │ N    │  ├── WhiteList[3]                                  │
│ 1036800 │   50.512 │  50.512 │ N    │  └── NearestNeighbor[4]                            │
└─────────┴──────────┴─────────┴──────┴────────────────────────────────────────────────────┘
# second
┌─────────┬──────────┬─────────┬──────┬───────────────────────────────────────────────────────┐
│ seeks   │ total_ms │ self_ms │ step │ query
tree                                            │
├─────────┼──────────┼─────────┼──────┼───────────────────────────────────────────────────────┤
│    7966 │  159.081 │  76.350 │ S    │  And[1]                                               │
│ 1000002 │   15.992 │  15.992 │ S    │  ├── Attribute
{int32, fs}[2]
timestamp_second: < range >  │
│ 1000002 │   15.992 │  15.991 │ N    │  ├── WhiteList[3]                                     │
│ 1000001 │   50.748 │  50.748 │ N    │  └── NearestNeighbor[4]                               │
└─────────┴──────────┴─────────┴──────┴───────────────────────────────────────────────────────┘
nofs
match
profiling
for thread  #0 (total time was 232.227 ms)
    ┌─────────┬──────────┬─────────┬──────┬────────────────────────────────────────────────────────────────┐
│ seeks   │ total_ms │ self_ms │ step │ query
tree                                                     │
├─────────┼──────────┼─────────┼──────┼────────────────────────────────────────────────────────────────┤
│    7966 │  232.227 │ 109.561 │ S    │  And[1]                                                        │
│ 1000002 │   42.270 │  42.270 │ S    │  ├── Attribute
{int32, lookup}[2]
timestamp_second_nofs: < range >  │
│ 1000001 │   30.292 │  30.292 │ N    │  ├── WhiteList[3]                                              │
│ 1000001 │   50.104 │  50.104 │ N    │  └── NearestNeighbor[4]                                        │
└─────────┴──────────┴─────────┴──────┴────────────────────────────────────────────────────────────────┘

SyntaxError: invalid character '┌' (U+250C) (1990059667.py, line 3)

In [202]:
### SLOW VERSION
combined = client.query(body={
    **trace.add_trace(request("timestamp_second_nofs")),
    "yql": f"""
                 select *
                 from sources *
                 where
                    (
                        (timestamp_day >= {get_timestamp()} AND (timestamp_second_nofs > {get_timestamp() + 100}))
                        AND (timestamp_day <= {get_timestamp() + 3000000} AND (timestamp_second_nofs < {get_timestamp() + 3000000 - 100}))
                        AND ({{targetHits: 1000, approximate: false}}nearestNeighbor(embedding, query_embedding))
                    )
                 """,
    "ranking.profile": "minimal",
    "hits": 100,
}).json
print(trace.inspect_trace(combined))

┌─────────┬─────────────┐
│ total   │ 1009.000 ms │
├─────────┼─────────────┤
│ query   │ 1008.000 ms │
│ summary │    0.000 ms │
│ other   │    1.000 ms │
└─────────┴─────────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │   1002.954 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 1002.954 ms
┌───────────────┬────────────┐
│ task          │ doc[0]     │
├───────────────┼────────────┤
│ global filter │   0.000 ms │
│ ann setup     │   0.000 ms │
│ matching      │ 951.808 ms │
│ first phase   │   0.008 ms │
│ second phase  │   0.000 ms │
└───────────────┴────────────┘
looking into node doc[0]
┌─────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp   │ event                                                       │
├─────────────┼────────────

In [196]:
### FAST VERSION
combined = client.query(body={
    **trace.add_trace(request("timestamp_second_nofs")),
    "yql": f"""
                 select *
                 from sources *
                 where
                    (
                        (timestamp_day >= {get_timestamp()})
                        AND (timestamp_day <= {get_timestamp() + 3000000})
                        AND ({{targetHits: 1000, approximate: false}}nearestNeighbor(embedding, query_embedding))
                    )
                 """,
    "ranking.profile": "minimal",
    "hits": 100,
}).json
print(trace.inspect_trace(combined))

┌─────────┬────────────┐
│ total   │ 554.000 ms │
├─────────┼────────────┤
│ query   │ 552.000 ms │
│ summary │   1.000 ms │
│ other   │   1.000 ms │
└─────────┴────────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │    549.691 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 549.691 ms
┌───────────────┬────────────┐
│ task          │ doc[0]     │
├───────────────┼────────────┤
│ global filter │   0.000 ms │
│ ann setup     │   0.000 ms │
│ matching      │ 535.814 ms │
│ first phase   │   0.001 ms │
│ second phase  │   0.000 ms │
└───────────────┴────────────┘
looking into node doc[0]
┌────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp  │ event                                                       │
├────────────┼───────────────────────

In [205]:
### CLEVER rewrite
combined = client.query(body={
    **trace.add_trace(request("timestamp_second_nofs")),
    "yql": f"""
                 select *
                 from sources *
                 where
                    (
                        (timestamp_day >= {get_timestamp()} AND timestamp_day <= {get_timestamp() + 3000000})
                        AND ({{targetHits: 1000, approximate: false}}nearestNeighbor(embedding, query_embedding))
                        AND ((timestamp_second_nofs > {get_timestamp() + 100}) AND (timestamp_second_nofs < {get_timestamp() + 3000000 - 100}))
                    )
                 """,
    "ranking.profile": "minimal",
    "hits": 100,
}).json
print(trace.inspect_trace(combined))

┌─────────┬────────────┐
│ total   │ 762.000 ms │
├─────────┼────────────┤
│ query   │ 760.000 ms │
│ summary │   0.000 ms │
│ other   │   2.000 ms │
└─────────┴────────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │    757.574 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 757.574 ms
┌───────────────┬────────────┐
│ task          │ doc[0]     │
├───────────────┼────────────┤
│ global filter │   0.000 ms │
│ ann setup     │   0.000 ms │
│ matching      │ 744.412 ms │
│ first phase   │   0.001 ms │
│ second phase  │   0.000 ms │
└───────────────┴────────────┘
looking into node doc[0]
┌────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp  │ event                                                       │
├────────────┼───────────────────────

In [201]:
## SIMPLY use second level fast search
combined = client.query(body={
    **trace.add_trace(request("timestamp_second_nofs")),
    "yql": f"""
                 select *
                 from sources *
                 where
                    (
                        (timestamp_second >= {get_timestamp()} AND timestamp_second <= {get_timestamp() + 3000000})
                        AND ({{targetHits: 1000, approximate: false}}nearestNeighbor(embedding, query_embedding))
                    )
                 """,
    "ranking.profile": "minimal",
    "hits": 100,
}).json
print(trace.inspect_trace(combined))

┌─────────┬────────────┐
│ total   │ 635.000 ms │
├─────────┼────────────┤
│ query   │ 632.000 ms │
│ summary │   2.000 ms │
│ other   │   1.000 ms │
└─────────┴────────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │    628.630 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 628.630 ms
┌───────────────┬────────────┐
│ task          │ doc[0]     │
├───────────────┼────────────┤
│ global filter │   0.000 ms │
│ ann setup     │   0.000 ms │
│ matching      │ 517.066 ms │
│ first phase   │   0.001 ms │
│ second phase  │   0.000 ms │
└───────────────┴────────────┘
looking into node doc[0]
┌────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp  │ event                                                       │
├────────────┼───────────────────────

In [207]:
### LET's do a brutal hack: in the ranking profile we should do a "filter" on precise time range:
### - if the value falls outside range let's set it to low value and the rank profile should filter out such hits
### - It is the first phase after all, the scores calculated should be cheap

In [250]:
app.get_schema("doc").add_rank_profile(
    RankProfile(
        name="range_filter_hack",
        inputs=[
            # How to write -infinity and +Infinity
            ("query(min_range_value)", "double", "-10000000000"),
            ("query(max_range_value)", "double", "10000000000"),
        ],
        first_phase=FirstPhaseRanking(
            expression="if ((query(min_range_value) >= attribute(timestamp_second_nofs) && attribute(timestamp_second_nofs) <= query(max_range_value)), 1, -1)",
            rank_score_drop_limit=-1.0,
        )
    )
)

In [251]:
print(app.get_schema("doc").schema_to_text)

schema doc {
    document doc {
        field id type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field timestamp_second_nofs type int {
            indexing: attribute
        }
        field timestamp_second type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field timestamp_hour type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field timestamp_day type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field embedding type tensor<float>(x[1]) {
            indexing: attribute
        }
        field lexical type string {
            indexing: index
            index: enable-bm25
        }
    }
    rank-profile minimal {
        first-phase {
            expression {
                1
  

In [252]:
vap.redeploy(vespa_docker, app)

Deploy status code: 200


Vespa(http://localhost, 8080)

In [223]:
min_range_value = get_timestamp()
max_range_value = min_range_value + 1 # so there should be two hits
combined = client.query(body={
    **request("timestamp_second_nofs"),
    "yql": f"""
                 select *
                 from sources *
                 where
                    (
                        (timestamp_second >= {min_range_value - 1} AND timestamp_second <= {max_range_value + 1})
                    )
                 """,
    "ranking.profile": "range_filter_hack", # changing to unranked, makes 4 docs to match
    "hits": 100,
    "input.query(min_range_value)": min_range_value,
    "input.query(max_range_value)": min_range_value,
}).json
combined

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::2072944',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::2072944',
    'relevance': 1.0,
    'source': 'test_content'},
   {'fields': {'documentid': 'id:doc:doc::2157537', 'sddocname': 'doc'},
    'id': 'id:doc:doc::2157537',
    'relevance': 1.0,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 10000000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 2},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.002,
  'searchtime': 0.004,
  'summaryfetchtime': 0.001}}

In [263]:
### Check if scoring is faster than filter evaluation on non fast-search attributes
min_range_value = get_timestamp()
max_range_value = min_range_value + 1 # so there should be two hits
request_body = {
    **request("timestamp_second_nofs"),
    "yql": f"""
                 select *
                 from sources *
                 where true
                 """,
    "ranking.profile": "range_filter_hack", # changing to unranked, makes 4 docs to match
    "hits": 1,
    "input.query(min_range_value)": min_range_value,
    "input.query(max_range_value)": min_range_value,
    "timeout": "55s",
}
# combined = client.query(body=trace.add_trace(request_body)).json
# print(trace.inspect_trace(combined))
combined = client.query(body=request_body).json
combined

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::38',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::38',
    'relevance': 1.0,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 10000000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 154131},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.507, 'searchtime': 0.508, 'summaryfetchtime': 0.0}}

In [271]:
### Check if matching is used on that attribute
min_range_value = get_timestamp()
max_range_value = min_range_value + 1 # so there should be two hits
request_body = {
    **request("timestamp_second_nofs"),
    "yql": f"""
                 select *
                 from sources *
                 where  timestamp_second_nofs > 0 AND timestamp_second_nofs < 100000000000
                 """,
    "ranking.profile": "unranked", # changing to unranked, makes 4 docs to match
    "hits": 1,
    # "input.query(min_range_value)": min_range_value,
    # "input.query(max_range_value)": min_range_value,
    "timeout": "55s",
}
combined = client.query(body=trace.add_trace(request_body)).json
print(trace.inspect_trace(combined))
# combined = client.query(body=request_body).json
# combined

┌─────────┬─────────────┐
│ total   │ 2116.000 ms │
├─────────┼─────────────┤
│ query   │ 2116.000 ms │
│ summary │    0.000 ms │
│ other   │    0.000 ms │
└─────────┴─────────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │   2114.064 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 2114.064 ms
┌───────────────┬─────────────┐
│ task          │ doc[0]      │
├───────────────┼─────────────┤
│ global filter │    0.000 ms │
│ ann setup     │    0.000 ms │
│ matching      │ 1557.606 ms │
│ first phase   │    0.000 ms │
│ second phase  │    0.000 ms │
└───────────────┴─────────────┘
looking into node doc[0]
┌─────────────┬─────────────────────────────────────────────────────────────┐
│ timestamp   │ event                                                       │
├─────────────┼───

In [206]:
#| label: yql_base
yql_base = """
           SELECT *
           FROM sources *
           WHERE
               (id
               > 1)
             AND (
               ({targetHits: 1000
               , approximate: false} nearestNeighbor(embedding
               , query_embedding))
              OR
               ({targetHits: 1000
               , defaultIndex: "lexical"} userInput(@query_str))
               ) \
           """
print(yql_base)


SELECT *
FROM sources *
WHERE
 (id> 1)
 AND (
   ({targetHits: 1000, approximate: false}nearestNeighbor(embedding, query_embedding))
   OR
   ({targetHits: 1000, defaultIndex: "lexical"}userInput(@query_str))
  )



In [186]:
request = {
    "yql": yql_base,
    "query_str": "27110 6334 10140 22335 22040 2716",
    "input.query(query_embedding)": [0.5],
    "presentation.timing": True,
    "hits": 1,
}
print(json.dumps(client.query(body=request).json, indent=2))

{
  "root": {
    "children": [
      {
        "fields": {
          "documentid": "id:doc:doc::96960",
          "sddocname": "doc"
        },
        "id": "id:doc:doc::96960",
        "relevance": 0.24059506636028924,
        "source": "test_content"
      }
    ],
    "coverage": {
      "coverage": 100,
      "documents": 100000,
      "full": true,
      "nodes": 1,
      "results": 1,
      "resultsFull": 1
    },
    "fields": {
      "totalCount": 5692
    },
    "id": "toplevel",
    "relevance": 1.0
  },
  "timing": {
    "querytime": 0.012,
    "searchtime": 0.013000000000000001,
    "summaryfetchtime": 0.0
  }
}


In [99]:
# Good we've found ~5692 docs

In [100]:
# Now let's try with tracing and ask vespa CLI to summarize the trace

In [173]:
import mycode.trace as trace

In [174]:
resp_base = client.query(body=trace.add_trace(request)).json

In [175]:
print(trace.inspect_trace(resp_base))

┌─────────┬───────────┐
│ total   │ 96.000 ms │
├─────────┼───────────┤
│ query   │ 94.000 ms │
│ summary │  1.000 ms │
│ other   │  1.000 ms │
└─────────┴───────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │     89.984 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 89.984 ms
┌───────────────┬───────────┐
│ task          │ doc[0]    │
├───────────────┼───────────┤
│ global filter │  0.000 ms │
│ ann setup     │  0.000 ms │
│ matching      │ 85.378 ms │
│ first phase   │  2.538 ms │
│ second phase  │  0.000 ms │
└───────────────┴───────────┘
looking into node doc[0]
┌───────────┬─────────────────────────────────────────────────────────────┐
│ timestamp │ event                                                       │
├───────────┼───────────────────────────────────────────

In [163]:
#| label: base_matching_summary
print(trace.get_matching_summary(trace.inspect_trace(resp_base)))

match profiling for thread #0 (total time was 91.847 ms)
┌────────┬──────────┬─────────┬──────┬────────────────────────────────────────────────┐
│ seeks  │ total_ms │ self_ms │ step │ query tree                                     │
├────────┼──────────┼─────────┼──────┼────────────────────────────────────────────────┤
│   5693 │   91.847 │   9.503 │ S    │  And[1]                                        │
│ 105690 │    2.840 │   2.628 │ S    │  ├── Attribute{int32,fs}[2] id:<range>         │
│  99998 │   77.513 │  12.364 │ N    │  ├── Or[3]                                     │
│ 100198 │    6.139 │   6.139 │ N    │  │   ├── NearestNeighbor[4]                    │
│  99998 │   59.010 │  35.133 │ N    │  │   └── WeakAnd[5]                            │
│  99998 │    3.928 │   3.906 │ N    │  │       ├── SourceBlender[6]                  │
│     37 │    0.022 │   0.022 │ N    │  │       │   └── MemoryTerm[7] lexical:27110   │
│  99998 │    3.959 │   3.944 │ N    │  │       ├── SourceBlend

In [111]:
# Above we see that weakAnd evaluated 99998 docs, which means that it can't prune matches.
# Now let's rewrite the query

In [205]:
#| label: yql_alt
yql_alt = """
          SELECT *
          FROM sources *
          WHERE
              (id
              > 1
            AND ({targetHits: 1000
              , approximate: false}
              nearestNeighbor(embedding
              , query_embedding))
             OR
              (id
              > 1
            AND ({targetHits: 1000
              , defaultIndex: "lexical"} userInput(@query_str)))) \
          """
print(yql_alt)


SELECT *
FROM sources *
WHERE
  (id> 1 AND ({targetHits: 1000, approximate: false}
              nearestNeighbor(embedding, query_embedding))
  OR
  (id> 1 AND ({targetHits: 1000, defaultIndex: "lexical"}userInput(@query_str))))



In [187]:
client.query(body={
    **request,
    "yql": yql_alt,
}).json

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::96960',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::96960',
    'relevance': 0.24030457979529288,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 100000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 5692},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.007,
  'searchtime': 0.009000000000000001,
  'summaryfetchtime': 0.0}}

In [115]:
resp_alt = client.query(body={
    **trace.add_trace(request),
    "yql": yql,
}).json
print(trace.inspect_trace(resp_alt))

┌─────────┬───────────┐
│ total   │ 35.000 ms │
├─────────┼───────────┤
│ query   │ 33.000 ms │
│ summary │  1.000 ms │
│ other   │  1.000 ms │
└─────────┴───────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │     26.625 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 26.625 ms
┌───────────────┬───────────┐
│ task          │ doc[0]    │
├───────────────┼───────────┤
│ global filter │  0.000 ms │
│ ann setup     │  0.000 ms │
│ matching      │ 21.827 ms │
│ first phase   │  2.696 ms │
│ second phase  │  0.000 ms │
└───────────────┴───────────┘
looking into node doc[0]
┌───────────┬─────────────────────────────────────────────────────────────┐
│ timestamp │ event                                                       │
├───────────┼───────────────────────────────────────────

In [166]:
#| label: alt_matching_summary
print(trace.get_matching_summary(trace.inspect_trace(resp_alt)))

match profiling for thread #0 (total time was 21.827 ms)
┌───────┬──────────┬─────────┬──────┬────────────────────────────────────────────────────┐
│ seeks │ total_ms │ self_ms │ step │ query tree                                         │
├───────┼──────────┼─────────┼──────┼────────────────────────────────────────────────────┤
│  5693 │   21.827 │   1.129 │ S    │  And[1]                                            │
│  5693 │   20.494 │   0.914 │ S    │  ├── Or[2]                                         │
│  5492 │   19.203 │   9.370 │ S    │  │   ├── And[3]                                    │
│ 99998 │    3.764 │   3.764 │ S    │  │   │   ├── Attribute{int32,fs}[4] id:<range>     │
│ 99998 │    6.070 │   6.070 │ N    │  │   │   └── NearestNeighbor[5]                    │
│   213 │    0.377 │   0.059 │ S    │  │   └── And[6]                                    │
│   213 │    0.249 │   0.073 │ S    │  │       ├── WeakAnd[7]                            │
│    38 │    0.027 │   0.013 │ S 

In [116]:
# above we see that weakAnd evaluated only 213 docs
# Which resulted in significantly lower latency: from ~100ms down to ~35ms.

In [162]:
print(trace.get_matching_summary(trace.inspect_trace(resp_alt)))

match profiling for thread #0 (total time was 21.827 ms)
┌───────┬──────────┬─────────┬──────┬────────────────────────────────────────────────────┐
│ seeks │ total_ms │ self_ms │ step │ query tree                                         │
├───────┼──────────┼─────────┼──────┼────────────────────────────────────────────────────┤
│  5693 │   21.827 │   1.129 │ S    │  And[1]                                            │
│  5693 │   20.494 │   0.914 │ S    │  ├── Or[2]                                         │
│  5492 │   19.203 │   9.370 │ S    │  │   ├── And[3]                                    │
│ 99998 │    3.764 │   3.764 │ S    │  │   │   ├── Attribute{int32,fs}[4] id:<range>     │
│ 99998 │    6.070 │   6.070 │ N    │  │   │   └── NearestNeighbor[5]                    │
│   213 │    0.377 │   0.059 │ S    │  │   └── And[6]                                    │
│   213 │    0.249 │   0.073 │ S    │  │       ├── WeakAnd[7]                            │
│    38 │    0.027 │   0.013 │ S 

In [199]:
#| label: and_query
yql_and = """
          SELECT *
          FROM sources *
          WHERE
              (id
              > 1)
            AND (
              ({targetHits: 1000
              , approximate: false} nearestNeighbor(embedding
              , query_embedding))
             OR
              ({targetHits: 1000
              , defaultIndex: "lexical"
              , grammar: "all"} userInput(@query_str))
              ) \
          """
request = {
    "yql": yql_and,
    "query_str": "27110 6334 10140 22335 22040 2716",
    "input.query(query_embedding)": [0.5],
    "presentation.timing": True,
    "hits": 1,
}
client.query(body=request).json

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::96960',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::96960',
    'relevance': 0.24059506636028924,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 100000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 5493},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.007, 'searchtime': 0.008, 'summaryfetchtime': 0.0}}

In [207]:
#| label: and_query
yql_no_filters = """
                 select *
                 from sources *
                 where (
                     ({targetHits: 1000
                     , approximate: false} nearestNeighbor(embedding
                     , query_embedding))
                    OR
                     ({targetHits: 1000
                     , defaultIndex: "lexical"} userInput(@query_str))
                     )
                 """
request = {
    **trace.add_trace(request),
    "yql": yql_no_filters,
    "query_str": "27110 6334 10140 22335 22040 2716",
    "input.query(query_embedding)": [0.5],
    "presentation.timing": True,
    "hits": 1,
}
resp_no_filters = client.query(body=request).json
print(trace.get_matching_summary(trace.inspect_trace(resp_no_filters)))

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::96960',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::96960',
    'relevance': 0.23972360666530013,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 100000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 5690},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.018000000000000002,
  'searchtime': 0.02,
  'summaryfetchtime': 0.001},
 'trace': {'children': [{'message': "Using query profile 'default' of type 'root'"},
   {'message': 'Resolved properties:\n'},
   {'message': "Invoking chain 'vespa' [com.yahoo.prelude.statistics.StatisticsSearcher@native -> com.yahoo.prelude.querytransform.PhrasingSearcher@vespa -> ... -> federation@native]"},
   {'children': [{'message': "Invoke searcher 'com.yahoo.prelude.statistics.StatisticsSearcher in native'",
      'timestamp': 0},
     {'message': "Invoke searcher 'com.yahoo.prelude.query

In [209]:
#| label: no_filters_matching_summary
print(trace.get_matching_summary(trace.inspect_trace(resp_no_filters)))

match profiling for thread #0 (total time was 4.903 ms)
┌───────┬──────────┬─────────┬──────┬────────────────────────────────────────────────┐
│ seeks │ total_ms │ self_ms │ step │ query tree                                     │
├───────┼──────────┼─────────┼──────┼────────────────────────────────────────────────┤
│  5691 │    4.903 │   1.046 │ S    │  And[1]                                        │
│  5691 │    3.670 │   0.942 │ S    │  ├── Or[2]                                     │
│  5491 │    1.762 │   1.762 │ S    │  │   ├── NearestNeighbor[3]                    │
│   213 │    0.965 │   0.059 │ S    │  │   └── WeakAnd[4]                            │
│    38 │    0.126 │   0.009 │ S    │  │       ├── SourceBlender[5]                  │
│    37 │    0.116 │   0.116 │ S    │  │       │   └── MemoryTerm[6] lexical:27110   │
│    25 │    0.121 │   0.011 │ S    │  │       ├── SourceBlender[7]                  │
│    24 │    0.109 │   0.109 │ S    │  │       │   └── MemoryTerm[8] lexic